# 02 — Spatial Orientation Forensics and Quality Audit

This notebook details the data forensics pipeline executed on the LiTS-17 cohort to detect spatial alignment anomalies, fix in-plane orientation flips, and audit 3D ROI containment.

### Objectives:
1. Verify 100% slice file integrity (0 missing/corrupted files).
2. Identify the 47 volumes requiring $180^\circ$ spatial rotation fixes.
3. Reconcile legacy $512\times 512$ denominator metric under-reporting.
4. Verify 100% tumor containment inside predicted-liver ROI bounding boxes.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rot180_volumes = [83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99] + list(range(101, 131))
identity_volumes = [v for v in range(131) if v not in rot180_volumes]

print(f"Volumes using Identity Transform (84 total): {identity_volumes[:10]}...")
print(f"Volumes requiring Rot180 Fix (47 total): {rot180_volumes[:10]}...")

## 1. Spatial Orientation Correction Map

Audit identified that raw Kaggle Part 2 and Hugging Face import packages contained a $180^\circ$ in-plane rotation discrepancy between CT slice images and segmentation labels.

In [ ]:
transform_counts = {"Identity": len(identity_volumes), "Rot180 Fix": len(rot180_volumes)}
plt.figure(figsize=(7, 4))
plt.bar(transform_counts.keys(), transform_counts.values(), color=["teal", "coral"])
plt.title("Volume Spatial Transform Distribution (131 Cohort Scans)", fontsize=13, fontweight="bold")
plt.ylabel("Number of Volumes")
for i, v in enumerate(transform_counts.values()):
    plt.text(i, v + 1, str(v), ha="center", fontweight="bold")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

## 2. 4x Tumor Burden Metric Reconciliation

- **Legacy Error**: Tumor burden percentage was computed using a $512\times 512$ pixel denominator ($262,144\text{ px}$) against $256\times 256$ native masks ($65,536\text{ px}$).
- **Reconciliation**: Native mask calculation adjusts true mean training tumor burden from `0.0251%` to **`0.1003%`**.

In [ ]:
legacy_burden = 0.0251
corrected_burden = legacy_burden * 4.0
print(f"Legacy 512x512 Denominator Mean Train Burden: {legacy_burden}%")
print(f"Corrected Native 256x256 Denominator Mean Train Burden: {corrected_burden:.4f}%")